
# Reconstrucción de plots de evaluación final

Esta notebook reconstruye **todos los plots de la sesión final** a partir del path de una carpeta de evaluación final ya generada, **sin volver a entrenar** y **sin sobrescribir** archivos originales.

## Qué reconstruye

### 1. Plots agregados por `artifact_type`
- `boxplots_<artifact_type>.png`
- `boxplot_<metrica>_<artifact_type>.png` (individuales)
- `best_validation_curves_<artifact_type>.png`
- `curva_exactitud_validacion_<artifact_type>.png`
- `curva_error_validacion_<artifact_type>.png`
- `curva_perdida_validacion_<artifact_type>.png`

### 2. Plots por corrida (opcional)
Reconstruidos desde cada `history.csv`:
- `plot_loss.png`
- `plot_accuracy.png`
- `plot_error.png`

## Requisitos que cumple
1. Reconstruir todas las imágenes a partir del path de una sesión final.
2. Permitir cambiar tamaños de letras, números y labels.
3. Guardar todo en un directorio nuevo de reconstrucción, sin tocar los plots originales.


In [1]:

# =========================
# IMPORTS
# =========================
from pathlib import Path
import copy
import datetime as dt
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


In [18]:

# =========================
# CONFIGURACION
# =========================
# Reemplaza este path por el de tu sesion de evaluacion final.
FINAL_EXPERIMENT_DIR = Path(r'runs/final_evaluations/2026-04-12-07-16-37-409003-tesis_goal2_auto-f62af822')

# Si es None, usa automaticamente los artifact types del metadata.
SELECTED_ARTIFACT_TYPES = None

# Carpeta donde se guardan las reconstrucciones.
# La notebook crea:
# <FINAL_EXPERIMENT_DIR>/_reconstructions/<RECONSTRUCTION_NAME>_<timestamp>/
RECONSTRUCTION_NAME = 'replot'

# Configuracion de tamanos y resolucion.
DEFAULT_PLOT_CONFIG = {
    'dpi': 320,

    'boxplots_combined_figsize': (26, 12),
    'boxplots_single_figsize': (8, 6.6),
    'curves_combined_figsize': (24, 7.2),
    'curves_single_figsize': (8.2, 6.2),
    'training_single_figsize': (8.5, 5.8),

    'suptitle_fontsize': 22,
    'title_fontsize': 18,
    'label_fontsize': 16,
    'tick_fontsize': 14,
    'legend_fontsize': 13,

    'x_tick_rotation': 15,
    'line_width': 2.4,
    'marker_size': 130,
    'grid_alpha': 0.28,
}

# Ejemplo: si queres modificar algo puntual, hacelo aca.
PLOT_CONFIG_OVERRIDES = {
    'suptitle_fontsize': 24,
    'title_fontsize': 20,
    'label_fontsize': 21,
    'tick_fontsize': 19,
    'legend_fontsize': 18,
    'dpi': 400,
    "validation_y_limits": {
    "val_error": (0.04, 0.12),
    "val_loss": (0.15, 0.6),
    # "val_acc": (0.96, 1.0),
    }
}


In [8]:

# =========================
# CONSTANTES Y HELPERS BASADOS EN LA NOTEBOOK ORIGINAL
# =========================
MODEL_DISPLAY_NAMES = {
    'OverfitNet': 'OverfitNet',
    'DropoutNet': 'Dropout',
    'DropConnectNet': 'DropConnect',
    'BoostDropoutNet': 'BoostDropout',
}

PAPER_MODEL_PALETTE = {
    'OverfitNet': '#4C78A8',
    'DropoutNet': '#F58518',
    'DropConnectNet': '#54A24B',
    'BoostDropoutNet': '#E45756',
}


def merge_plot_config(custom_config=None):
    cfg = copy.deepcopy(DEFAULT_PLOT_CONFIG)
    if custom_config:
        cfg.update(custom_config)
    return cfg



def read_json(path):
    path = Path(path)
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)



def write_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)



def ensure_unique_directory(path):
    path = Path(path)
    if not path.exists():
        path.mkdir(parents=True, exist_ok=False)
        return path

    parent = path.parent
    stem = path.name
    idx = 1
    while True:
        candidate = parent / f'{stem}_{idx:02d}'
        if not candidate.exists():
            candidate.mkdir(parents=True, exist_ok=False)
            return candidate
        idx += 1



def make_reconstruction_root(final_experiment_dir, reconstruction_name='replot'):
    final_experiment_dir = Path(final_experiment_dir)
    timestamp = dt.datetime.now().strftime('%Y-%m-%d-%H-%M-%S-%f')
    base_dir = final_experiment_dir / '_reconstructions'
    base_dir.mkdir(parents=True, exist_ok=True)
    requested = base_dir / f'{reconstruction_name}_{timestamp}'
    return ensure_unique_directory(requested)



def _ordered_architectures_from_df(df):
    preferred = ['OverfitNet', 'DropoutNet', 'DropConnectNet', 'BoostDropoutNet']
    present = [name for name in preferred if name in set(df['architecture'].unique())]
    remaining = [name for name in sorted(df['architecture'].unique()) if name not in present]
    return present + remaining



def _pretty_architecture_label(name):
    return MODEL_DISPLAY_NAMES.get(name, name)



def _spanish_metric_title(metric_key):
    title_map = {
        'test_acc': '',
        'test_loss': '',
        'test_error': '',
        'val_acc': '',
        'val_loss': '',
        'val_error': '',
        'best_epoch': '',
        'training_time_end_to_end_minutes': 'Tiempo de entrenamiento end-to-end',
        'training_loop_time_minutes': 'Tiempo del loop de entrenamiento',
    }
    return title_map.get(metric_key, metric_key)



def _spanish_metric_ylabel(metric_key):
    ylabel_map = {
        'test_acc': 'Exactitud',
        'test_loss': 'Pérdida',
        'test_error': 'Error',
        'val_acc': 'Exactitud',
        'val_loss': 'Pérdida',
        'val_error': 'Error',
        'best_epoch': 'Época',
        'training_time_end_to_end_minutes': 'Minutos',
        'training_loop_time_minutes': 'Minutos',
    }
    return ylabel_map.get(metric_key, metric_key)



def _metric_slug(metric_key):
    slug_map = {
        'test_acc': 'exactitud_test',
        'test_loss': 'perdida_test',
        'test_error': 'error_test',
        'val_acc': 'exactitud_validacion',
        'val_loss': 'perdida_validacion',
        'val_error': 'error_validacion',
        'best_epoch': 'mejor_epoca',
        'training_time_end_to_end_minutes': 'tiempo_entrenamiento_end_to_end',
        'training_loop_time_minutes': 'tiempo_loop_entrenamiento',
    }
    return slug_map.get(metric_key, metric_key)


In [4]:

# =========================
# CARGA DE LA SESION FINAL
# =========================

def get_selected_artifact_types_from_metadata(final_experiment_dir):
    metadata_path = Path(final_experiment_dir) / 'final_experiment_metadata.json'
    if not metadata_path.exists():
        raise FileNotFoundError(f'No existe: {metadata_path}')

    metadata = read_json(metadata_path)
    artifact_types = metadata.get('selected_artifact_types')
    if artifact_types is None:
        artifact_types = sorted([
            p.name for p in Path(final_experiment_dir).iterdir()
            if p.is_dir() and not p.name.startswith('_')
        ])
    return artifact_types, metadata



def load_existing_final_detail_df(final_experiment_dir, artifact_type):
    final_experiment_dir = Path(final_experiment_dir)
    category_dir = final_experiment_dir / artifact_type

    detail_json = category_dir / f'final_results_detail_{artifact_type}.json'
    detail_csv = category_dir / f'final_results_detail_{artifact_type}.csv'

    if detail_json.exists():
        detail_df = pd.read_json(detail_json)
    elif detail_csv.exists():
        detail_df = pd.read_csv(detail_csv)
    else:
        raise FileNotFoundError(
            f'No se encontró final_results_detail para {artifact_type} en {category_dir}'
        )

    # Repara training_dir si el experimento fue movido a otro lugar.
    if 'training_id' in detail_df.columns:
        repaired_dirs = []
        for _, row in detail_df.iterrows():
            current = None
            if 'training_dir' in detail_df.columns and pd.notna(row.get('training_dir', np.nan)):
                current = Path(str(row['training_dir']))

            fallback = category_dir / str(row['training_id'])

            if current is not None and current.exists():
                repaired_dirs.append(str(current.resolve()))
            elif fallback.exists():
                repaired_dirs.append(str(fallback.resolve()))
            else:
                repaired_dirs.append(str(current) if current is not None else str(fallback))

        detail_df['training_dir'] = repaired_dirs

    if 'test_error' not in detail_df.columns and 'test_acc' in detail_df.columns:
        detail_df['test_error'] = 1.0 - pd.to_numeric(detail_df['test_acc'], errors='coerce')

    if 'val_error' not in detail_df.columns and 'val_acc' in detail_df.columns:
        detail_df['val_error'] = 1.0 - pd.to_numeric(detail_df['val_acc'], errors='coerce')

    return detail_df



def load_history_df(training_dir):
    history_path = Path(training_dir) / 'history.csv'
    if not history_path.exists():
        raise FileNotFoundError(f'No se encontró history.csv en {training_dir}')
    return pd.read_csv(history_path)


In [10]:

# =========================
# PLOTTING PARAMETRIZABLE
# =========================

def _set_tick_fontsizes(ax, cfg):
    ax.tick_params(axis='both', labelsize=cfg['tick_fontsize'])



def _draw_publication_boxplot_reconstructed(ax, df, metric, ylabel, title, cfg):
    order = _ordered_architectures_from_df(df)
    data = [
        pd.to_numeric(df.loc[df['architecture'] == arch, metric], errors='coerce').dropna().values
        for arch in order
    ]
    labels = [_pretty_architecture_label(arch) for arch in order]
    colors = [PAPER_MODEL_PALETTE.get(arch, '#4C78A8') for arch in order]

    bp = ax.boxplot(
        data,
        patch_artist=True,
        tick_labels=labels,
        widths=0.55,
        showfliers=False,
        medianprops={'color': '#1A1A1A', 'linewidth': 1.8},
        whiskerprops={'linewidth': 1.2},
        capprops={'linewidth': 1.2},
        boxprops={'linewidth': 1.2},
    )

    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.72)
        patch.set_edgecolor('#222222')

    rng = np.random.default_rng(12345)
    for idx, (_, values, color) in enumerate(zip(order, data, colors), start=1):
        if len(values) == 0:
            continue
        x = rng.normal(loc=idx, scale=0.045, size=len(values))
        ax.scatter(x, values, s=40, alpha=0.65, color=color, edgecolor='white', linewidth=0.45, zorder=3)
        ax.scatter([idx], [np.mean(values)], marker='D', s=68, color='#111111', edgecolor='white', linewidth=0.8, zorder=4)

    ax.set_title(title, fontsize=cfg['title_fontsize'], pad=10, fontweight='semibold')
    ax.set_ylabel(ylabel, fontsize=cfg['label_fontsize'])
    ax.grid(True, axis='y', linestyle='--', alpha=cfg['grid_alpha'])
    ax.set_axisbelow(True)
    ax.tick_params(axis='x', rotation=cfg['x_tick_rotation'])
    _set_tick_fontsizes(ax, cfg)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)



def plot_final_boxplots_reconstructed(detail_df, artifact_type, save_dir, plot_config=None):
    if detail_df.empty:
        return None

    cfg = merge_plot_config(plot_config)
    plt.style.use('seaborn-v0_8-whitegrid')
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    metric_specs = [
        'test_acc',
        'test_error',
        'test_loss',
        'val_loss',
        'val_error',
        'best_epoch',
        'training_time_end_to_end_minutes',
        'training_loop_time_minutes',
    ]

    fig, axes = plt.subplots(2, 4, figsize=cfg['boxplots_combined_figsize'])

    for ax, metric in zip(axes.ravel(), metric_specs):
        title = _spanish_metric_title(metric)
        ylabel = _spanish_metric_ylabel(metric)
        _draw_publication_boxplot_reconstructed(ax, detail_df, metric, ylabel, title, cfg)

    fig.suptitle('Boxplots de evaluación final', fontsize=cfg['suptitle_fontsize'], fontweight='bold', y=0.98)
    fig.tight_layout(rect=[0, 0, 1, 0.97])

    out_path = save_dir / f'boxplots_{artifact_type}.png'
    fig.savefig(out_path, dpi=cfg['dpi'], bbox_inches='tight')
    plt.close(fig)

    individual_paths = []
    for metric in metric_specs:
        fig_single, ax_single = plt.subplots(figsize=cfg['boxplots_single_figsize'])
        title = _spanish_metric_title(metric)
        ylabel = _spanish_metric_ylabel(metric)
        _draw_publication_boxplot_reconstructed(ax_single, detail_df, metric, ylabel, title, cfg)
        fig_single.tight_layout()
        single_path = save_dir / f'boxplot_{_metric_slug(metric)}_{artifact_type}.png'
        fig_single.savefig(single_path, dpi=cfg['dpi'], bbox_inches='tight')
        plt.close(fig_single)
        individual_paths.append(str(single_path))

    return {
        'combined_plot': str(out_path),
        'individual_plots': individual_paths,
    }



def plot_best_final_validation_curves_reconstructed(detail_df, artifact_type, save_dir, plot_config=None):
    if detail_df.empty:
        return None

    cfg = merge_plot_config(plot_config)
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    order = _ordered_architectures_from_df(detail_df)
    best_rows = []

    for arch in order:
        arch_df = detail_df.loc[detail_df['architecture'] == arch].copy()
        arch_df = arch_df.dropna(subset=['val_loss'])
        if arch_df.empty:
            continue

        best_row = arch_df.sort_values(['val_loss', 'test_loss', 'experiment_seed'], ascending=[True, True, True]).iloc[0]
        best_rows.append(best_row)

    if not best_rows:
        return None

    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(1, 3, figsize=cfg['curves_combined_figsize'])

    curve_data = {
        'val_acc': [],
        'val_error': [],
        'val_loss': [],
    }

    for row in best_rows:
        history_path = Path(row['training_dir']) / 'history.csv'
        if not history_path.exists():
            print(f'⚠️ Se omite una curva porque no existe: {history_path}')
            continue

        history_df = pd.read_csv(history_path)
        epochs = np.arange(1, len(history_df) + 1)
        color = PAPER_MODEL_PALETTE.get(row['architecture'], '#4C78A8')
        label = f"{_pretty_architecture_label(row['architecture'])}" # (seed={int(row['experiment_seed'])})"

        val_acc = pd.to_numeric(history_df['val_acc'], errors='coerce')
        val_error = 1.0 - val_acc
        val_loss = pd.to_numeric(history_df['val_loss'], errors='coerce')

        curve_data['val_acc'].append((epochs, val_acc, color, label, row))
        curve_data['val_error'].append((epochs, val_error, color, label, row))
        curve_data['val_loss'].append((epochs, val_loss, color, label, row))

        axes[0].plot(epochs, val_acc, linewidth=cfg['line_width'], color=color, label=label)
        axes[1].plot(epochs, val_error, linewidth=cfg['line_width'], color=color, label=label)
        axes[2].plot(epochs, val_loss, linewidth=cfg['line_width'], color=color, label=label)

        best_epoch = row.get('best_epoch')
        if pd.notna(best_epoch) and int(best_epoch) <= len(history_df):
            best_epoch = int(best_epoch)
            axes[0].scatter(best_epoch, val_acc.iloc[best_epoch - 1], s=cfg['marker_size'], color=color, edgecolor='white', linewidth=0.8, zorder=4)
            axes[1].scatter(best_epoch, val_error.iloc[best_epoch - 1], s=cfg['marker_size'], color=color, edgecolor='white', linewidth=0.8, zorder=4)
            axes[2].scatter(best_epoch, val_loss.iloc[best_epoch - 1], s=cfg['marker_size'], color=color, edgecolor='white', linewidth=0.8, zorder=4)

    axes[0].set_title('', fontsize=cfg['title_fontsize'], fontweight='semibold')
    axes[1].set_title('', fontsize=cfg['title_fontsize'], fontweight='semibold')
    axes[2].set_title('', fontsize=cfg['title_fontsize'], fontweight='semibold')

    axes[0].set_xlabel('Época', fontsize=cfg['label_fontsize'])
    axes[1].set_xlabel('Época', fontsize=cfg['label_fontsize'])
    axes[2].set_xlabel('Época', fontsize=cfg['label_fontsize'])

    axes[0].set_ylabel('Exactitud', fontsize=cfg['label_fontsize'])
    axes[1].set_ylabel('Error', fontsize=cfg['label_fontsize'])
    axes[2].set_ylabel('Pérdida', fontsize=cfg['label_fontsize'])

    y_limits = cfg.get("validation_y_limits", {})
    
    if y_limits.get("val_acc") is not None:
        axes[0].set_ylim(*y_limits["val_acc"])
    
    if y_limits.get("val_error") is not None:
        axes[1].set_ylim(*y_limits["val_error"])
    
    if y_limits.get("val_loss") is not None:
        axes[2].set_ylim(*y_limits["val_loss"])

    for ax in axes:
        ax.grid(True, linestyle='--', alpha=cfg['grid_alpha'])
        ax.set_axisbelow(True)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        _set_tick_fontsizes(ax, cfg)


    axes[2].legend(frameon=False, fontsize=cfg['legend_fontsize'], loc='best')

    fig.suptitle('Curvas de validación finales', fontsize=cfg['suptitle_fontsize'], fontweight='bold', y=0.98)
    fig.tight_layout(rect=[0, 0, 1, 0.96])

    out_path = save_dir / f'best_validation_curves_{artifact_type}.png'
    fig.savefig(out_path, dpi=cfg['dpi'], bbox_inches='tight')
    plt.close(fig)

    individual_paths = []
    individual_specs = [
        ('val_acc', '', 'Exactitud', f'curva_exactitud_validacion_{artifact_type}.png'),
        ('val_error', '', 'Error', f'curva_error_validacion_{artifact_type}.png'),
        ('val_loss', '', 'Pérdida', f'curva_perdida_validacion_{artifact_type}.png'),
    ]

    for metric_key, title, ylabel, filename in individual_specs:
        fig_single, ax_single = plt.subplots(figsize=cfg['curves_single_figsize'])

        for epochs, values, color, label, row in curve_data[metric_key]:
            ax_single.plot(epochs, values, linewidth=cfg['line_width'], color=color, label=label)
            history_path = Path(row['training_dir']) / 'history.csv'
            history_df = pd.read_csv(history_path)
            best_epoch = row.get('best_epoch')

            if pd.notna(best_epoch) and int(best_epoch) <= len(history_df):
                best_epoch = int(best_epoch)
                ax_single.scatter(best_epoch, values.iloc[best_epoch - 1], s=cfg['marker_size'], color=color, edgecolor='white', linewidth=0.8, zorder=4)

        ax_single.set_title(title, fontsize=cfg['title_fontsize'], fontweight='semibold')
        ax_single.set_xlabel('Época', fontsize=cfg['label_fontsize'])
        ax_single.set_ylabel(ylabel, fontsize=cfg['label_fontsize'])
        ax_single.grid(True, linestyle='--', alpha=cfg['grid_alpha'])
        ax_single.set_axisbelow(True)
        ax_single.spines['top'].set_visible(False)
        ax_single.spines['right'].set_visible(False)
        ax_single.legend(frameon=False, fontsize=cfg['legend_fontsize'], loc='best')
        _set_tick_fontsizes(ax_single, cfg)

        if y_limits.get(metric_key) is not None:
            ax_single.set_ylim(*y_limits[metric_key])

        fig_single.tight_layout()
        single_path = save_dir / filename
        fig_single.savefig(single_path, dpi=cfg['dpi'], bbox_inches='tight')
        plt.close(fig_single)
        individual_paths.append(str(single_path))

    return {
        'combined_plot': str(out_path),
        'individual_plots': individual_paths,
    }



def plot_single_training_metric(history_df, metric, save_dir, plot_config=None):
    cfg = merge_plot_config(plot_config)
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    epochs = np.arange(1, len(history_df) + 1)
    fig, ax = plt.subplots(figsize=cfg['training_single_figsize'])

    if metric == 'loss':
        ax.plot(epochs, pd.to_numeric(history_df['train_loss'], errors='coerce'), label='Train Loss', linewidth=cfg['line_width'])
        ax.plot(epochs, pd.to_numeric(history_df['val_loss'], errors='coerce'), label='Validation Loss', linestyle='--', linewidth=cfg['line_width'])
        ax.set_ylabel('Loss', fontsize=cfg['label_fontsize'])
        ax.set_title('Loss Curves', fontsize=cfg['title_fontsize'])
        filename = 'plot_loss.png'
    elif metric == 'accuracy':
        ax.plot(epochs, pd.to_numeric(history_df['train_acc'], errors='coerce'), label='Train Accuracy', linewidth=cfg['line_width'])
        ax.plot(epochs, pd.to_numeric(history_df['val_acc'], errors='coerce'), label='Validation Accuracy', linestyle='--', linewidth=cfg['line_width'])
        ax.set_ylabel('Accuracy', fontsize=cfg['label_fontsize'])
        ax.set_title('Accuracy Curves', fontsize=cfg['title_fontsize'])
        filename = 'plot_accuracy.png'
    elif metric == 'error':
        train_error = 1.0 - pd.to_numeric(history_df['train_acc'], errors='coerce')
        val_error = 1.0 - pd.to_numeric(history_df['val_acc'], errors='coerce')
        ax.plot(epochs, train_error, label='Train Error', linewidth=cfg['line_width'])
        ax.plot(epochs, val_error, label='Validation Error', linestyle='--', linewidth=cfg['line_width'])
        ax.set_ylabel('Error', fontsize=cfg['label_fontsize'])
        ax.set_title('Error Curves', fontsize=cfg['title_fontsize'])
        filename = 'plot_error.png'
    else:
        raise ValueError("metric debe ser 'loss', 'accuracy' o 'error'")

    ax.set_xlabel('Epoch', fontsize=cfg['label_fontsize'])
    ax.legend(frameon=False, fontsize=cfg['legend_fontsize'])
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    _set_tick_fontsizes(ax, cfg)
    fig.tight_layout()

    save_path = save_dir / filename
    fig.savefig(save_path, dpi=cfg['dpi'], bbox_inches='tight')
    plt.close(fig)
    return str(save_path)


In [6]:

# =========================
# RECONSTRUCCION COMPLETA
# =========================

def reconstruct_per_run_training_plots(detail_df, artifact_type, save_dir, plot_config=None):
    if detail_df.empty:
        return []

    save_dir = Path(save_dir)
    outputs = []

    for _, row in detail_df.iterrows():
        training_dir = Path(row['training_dir'])
        history_path = training_dir / 'history.csv'
        if not history_path.exists():
            print(f'⚠️ Se omite corrida sin history.csv: {history_path}')
            continue

        history_df = pd.read_csv(history_path)
        training_id = str(row.get('training_id', training_dir.name))
        run_output_dir = save_dir / artifact_type / 'training_runs' / training_id
        run_output_dir.mkdir(parents=True, exist_ok=True)

        loss_path = plot_single_training_metric(history_df, 'loss', run_output_dir, plot_config=plot_config)
        acc_path = plot_single_training_metric(history_df, 'accuracy', run_output_dir, plot_config=plot_config)
        err_path = plot_single_training_metric(history_df, 'error', run_output_dir, plot_config=plot_config)

        outputs.append({
            'training_id': training_id,
            'source_training_dir': str(training_dir),
            'output_dir': str(run_output_dir),
            'plots': {
                'loss': loss_path,
                'accuracy': acc_path,
                'error': err_path,
            },
        })

    return outputs



def reconstruct_final_evaluation_plots(
    final_experiment_dir,
    selected_artifact_types=None,
    reconstruction_name='replot',
    plot_config=None,
    rebuild_boxplots=True,
    rebuild_validation_curves=True,
    rebuild_per_run_training_plots=True,
):
    final_experiment_dir = Path(final_experiment_dir)
    if not final_experiment_dir.exists():
        raise FileNotFoundError(f'No existe FINAL_EXPERIMENT_DIR: {final_experiment_dir}')

    artifact_types_from_metadata, source_metadata = get_selected_artifact_types_from_metadata(final_experiment_dir)
    if selected_artifact_types is None:
        selected_artifact_types = artifact_types_from_metadata

    reconstruction_root = make_reconstruction_root(final_experiment_dir, reconstruction_name=reconstruction_name)
    cfg = merge_plot_config(plot_config)

    result = {
        'source_final_experiment_dir': str(final_experiment_dir.resolve()),
        'reconstruction_root': str(reconstruction_root.resolve()),
        'selected_artifact_types': list(selected_artifact_types),
        'plot_config': cfg,
        'generated_at': dt.datetime.now().isoformat(),
        'categories': {},
    }

    for artifact_type in selected_artifact_types:
        print('\n' + '=' * 100)
        print(f'🔁 Reconstruyendo artifact_type={artifact_type}')
        print('=' * 100)

        category_output_dir = reconstruction_root / artifact_type
        category_output_dir.mkdir(parents=True, exist_ok=True)
        detail_df = load_existing_final_detail_df(final_experiment_dir, artifact_type)

        category_result = {
            'output_dir': str(category_output_dir.resolve()),
            'num_rows': int(len(detail_df)),
            'boxplots': None,
            'validation_curves': None,
            'per_run_training_plots': None,
        }

        if rebuild_boxplots:
            boxplot_dir = category_output_dir / 'boxplots'
            boxplot_dir.mkdir(parents=True, exist_ok=True)
            category_result['boxplots'] = plot_final_boxplots_reconstructed(
                detail_df=detail_df,
                artifact_type=artifact_type,
                save_dir=boxplot_dir,
                plot_config=cfg,
            )

        if rebuild_validation_curves:
            curves_dir = category_output_dir / 'validation_curves'
            curves_dir.mkdir(parents=True, exist_ok=True)
            category_result['validation_curves'] = plot_best_final_validation_curves_reconstructed(
                detail_df=detail_df,
                artifact_type=artifact_type,
                save_dir=curves_dir,
                plot_config=cfg,
            )

        if rebuild_per_run_training_plots:
            training_plots_dir = category_output_dir / 'per_run_plots'
            training_plots_dir.mkdir(parents=True, exist_ok=True)
            category_result['per_run_training_plots'] = reconstruct_per_run_training_plots(
                detail_df=detail_df,
                artifact_type=artifact_type,
                save_dir=training_plots_dir,
                plot_config=cfg,
            )

        result['categories'][artifact_type] = category_result

    write_json(result, reconstruction_root / 'reconstruction_metadata.json')

    print('\n' + '=' * 100)
    print('✅ RECONSTRUCCION FINALIZADA')
    print('=' * 100)
    print(f'📁 Salida: {reconstruction_root}')
    print('⚠️ Los archivos originales del experimento no fueron modificados.')

    return result


In [17]:

# =========================
# EJECUCION
# =========================
plot_config = merge_plot_config(PLOT_CONFIG_OVERRIDES)

# Descomenta cuando hayas configurado FINAL_EXPERIMENT_DIR.
results = reconstruct_final_evaluation_plots(
    final_experiment_dir=FINAL_EXPERIMENT_DIR,
    selected_artifact_types=SELECTED_ARTIFACT_TYPES,
    reconstruction_name=RECONSTRUCTION_NAME,
    plot_config=plot_config,
    rebuild_boxplots=False,
    rebuild_validation_curves=True,
    rebuild_per_run_training_plots=True,
)
results



🔁 Reconstruyendo artifact_type=best_model

🔁 Reconstruyendo artifact_type=early_stopping

🔁 Reconstruyendo artifact_type=last_model

✅ RECONSTRUCCION FINALIZADA
📁 Salida: runs/final_evaluations/2026-04-12-07-16-37-409003-tesis_goal2_auto-f62af822/_reconstructions/replot_2026-04-29-20-46-05-366353
⚠️ Los archivos originales del experimento no fueron modificados.


{'source_final_experiment_dir': '/mnt/d2ddcf1e-b595-473f-a2bc-7304fcae012e/tes/tesis-project-repository/runs/final_evaluations/2026-04-12-07-16-37-409003-tesis_goal2_auto-f62af822',
 'reconstruction_root': '/mnt/d2ddcf1e-b595-473f-a2bc-7304fcae012e/tes/tesis-project-repository/runs/final_evaluations/2026-04-12-07-16-37-409003-tesis_goal2_auto-f62af822/_reconstructions/replot_2026-04-29-20-46-05-366353',
 'selected_artifact_types': ['best_model', 'early_stopping', 'last_model'],
 'plot_config': {'dpi': 400,
  'boxplots_combined_figsize': (26, 12),
  'boxplots_single_figsize': (8, 6.6),
  'curves_combined_figsize': (24, 7.2),
  'curves_single_figsize': (8.2, 6.2),
  'training_single_figsize': (8.5, 5.8),
  'suptitle_fontsize': 24,
  'title_fontsize': 20,
  'label_fontsize': 21,
  'tick_fontsize': 19,
  'legend_fontsize': 18,
  'x_tick_rotation': 15,
  'line_width': 2.4,
  'marker_size': 130,
  'grid_alpha': 0.28,
  'validation_y_limits': {'val_error': (0.04, 0.12), 'val_loss': (0.15, 0.6


## Dónde se guarda la salida

La notebook **nunca sobrescribe** los plots originales del experimento.

Todo se guarda en una carpeta nueva:

```text
<FINAL_EXPERIMENT_DIR>/_reconstructions/<RECONSTRUCTION_NAME>_<timestamp>/
```

Estructura esperada de salida:

```text
_reconstructions/
  replot_YYYY-mm-dd-HH-MM-SS-ffffff/
    reconstruction_metadata.json
    best_model/
      boxplots/
      validation_curves/
      per_run_plots/
    early_stopping/
      ...
    last_model/
      ...
```

## Qué tocar si querés letras más grandes

En `PLOT_CONFIG_OVERRIDES` podés cambiar, por ejemplo:

```python
PLOT_CONFIG_OVERRIDES = {
    'suptitle_fontsize': 24,
    'title_fontsize': 20,
    'label_fontsize': 18,
    'tick_fontsize': 16,
    'legend_fontsize': 15,
    'dpi': 400,
}
```
